# 🏙️ Project 38 — Urban Expansion Analysis

## 1. Introduce the Project

In Project 20, we compared land cover to monitor **deforestation**.

Now we'll use a similar idea to study **urban expansion**.

The basic question is:

> **Which areas were not urban before, but became urban later?**

For example:

```text
2015                         2025

🌳 🌳 🌳 🌳                  🌳 🌳 🏠 🏠
🌳 🏠 🌳 🌳        →         🌳 🏠 🏠 🏠
🌳 🌳 🌳 🌳                  🌳 🌳 🏠 🏠
```

The newly urbanized areas are the pixels that changed:

```text
Non-urban → Urban
```

Our final output will be a **map of newly urbanized areas**.

---

# 2. Big Picture Workflow

```text
        Land Cover Year 1
               │
               ▼
        Urban Mask 1
               │
               │
               │ Compare
               │
               ▼
        Urban Mask 2
               │
               ▼
        Urban Mask 2
        minus Urban Mask 1
               │
               ▼
     Newly Urbanized Areas
```

The important idea is:

> **We are detecting a transition, not simply mapping urban areas.**

---

# 3. Main Concept — Land Cover Comparison

We need a land-cover dataset that tells us what type of land each pixel represents.

We'll use:

```text
ESA WorldCover
```

This is the same land-cover dataset we used earlier.

The important class for this project is:

```text
Built-up → class 50
```

So we can create:

```text
Urban  = land cover == 50
```

---

# 4. What Does "Urban Expansion" Mean Here?

Suppose we have:

### Year 1

```text
Pixel A → Forest
Pixel B → Cropland
Pixel C → Built-up
```

### Year 2

```text
Pixel A → Built-up
Pixel B → Built-up
Pixel C → Built-up
```

Then:

```text
A: Forest → Built-up   ✅ New urban
B: Cropland → Built-up ✅ New urban
C: Built-up → Built-up ❌ Existing urban
```

This distinction is very important.

We don't want **all urban areas**.

We specifically want:

> **Newly urbanized areas.**

---

# 5. Important Limitation Before We Start

There is one important issue with the simple educational approach.

ESA WorldCover provides global land-cover maps for specific years, but it is **not an annual urban-growth dataset**.

Therefore, we should choose two years for which the dataset has suitable WorldCover products rather than pretending that we have continuous annual urban observations.

For this lesson, we'll use **2020 and 2021** as the comparison years.

This is a relatively short period, but it's appropriate for learning the workflow.

---

# 6. Step 1 — Define the Study Area

Let's use a smaller urban area rather than the large Black Forest polygon.

For example, you can study **Frankfurt and its surrounding area**:

```javascript
var studyArea = ee.Geometry.Polygon([
  [
    [8.35, 50.00],
    [8.90, 50.00],
    [8.90, 50.35],
    [8.35, 50.35],
    [8.35, 50.00]
  ]
]);
```

This gives us an area where urban and non-urban land can be compared.

---

# 7. Step 2 — Load the Two Land-Cover Images

We'll use the ESA WorldCover products:

```javascript
var worldcover2020 = ee.ImageCollection("ESA/WorldCover/v100")
  .first();

var worldcover2021 = ee.ImageCollection("ESA/WorldCover/v200")
  .first();
```

Here we are using two WorldCover versions representing different years.

---

# 8. Step 3 — Clip Them

```javascript
var lc2020 = worldcover2020.clip(studyArea);
var lc2021 = worldcover2021.clip(studyArea);
```

Now both datasets cover only our study area.

---

# 9. Step 4 — Create the Urban Masks

Remember:

```text
Built-up = class 50
```

So:

```javascript
var urban2020 = lc2020.eq(50);
var urban2021 = lc2021.eq(50);
```

Now each image contains essentially:

```text
1 → Urban
0 → Not urban
```

This is called a **binary mask**.

---

# 10. Step 5 — Find Newly Urbanized Areas

This is the most important line of the project:

```javascript
var newUrban = urban2021.and(urban2020.not());
```

Let's understand it slowly.

We want:

```text
2021 = Urban
AND
2020 = NOT Urban
```

So:

```text
2020        2021        Result

0           0           0
0           1           1  ← NEW URBAN
1           0           0
1           1           0
```

Only the transition:

```text
Non-urban → Urban
```

becomes `1`.

---

# 11. Why Not Just Subtract the Images?

You might think:

```javascript
urban2021.subtract(urban2020)
```

That would produce:

```text
2020    2021    Difference

0       0       0
0       1       +1  ← new urban
1       0       -1
1       1        0
```

This is actually useful for **change analysis**.

But here we're interested specifically in:

> **new urbanization**

So the logical condition:

```javascript
urban2021.and(urban2020.not())
```

is clearer.

---

# 12. Step 6 — Display Existing Urban Areas

Let's first display the urban area in 2020:

```javascript
Map.addLayer(
  urban2020.selfMask(),
  {},
  "Urban 2020"
);
```

Then 2021:

```javascript
Map.addLayer(
  urban2021.selfMask(),
  {},
  "Urban 2021"
);
```

This allows you to visually compare the two.

---

# 13. Step 7 — Display Newly Urbanized Areas

```javascript
Map.addLayer(
  newUrban.selfMask(),
  {},
  "Newly Urbanized Areas"
);
```

This is our main output.

---

# 14. Complete Code

```javascript
// ==========================================
// Project 38 — Urban Expansion Analysis
// ==========================================

// 1. Define study area
var studyArea = ee.Geometry.Polygon([
  [
    [8.35, 50.00],
    [8.90, 50.00],
    [8.90, 50.35],
    [8.35, 50.35],
    [8.35, 50.00]
  ]
]);

// 2. Load WorldCover 2020
var worldcover2020 = ee.ImageCollection("ESA/WorldCover/v100")
  .first();

// 3. Load WorldCover 2021
var worldcover2021 = ee.ImageCollection("ESA/WorldCover/v200")
  .first();

// 4. Clip to study area
var lc2020 = worldcover2020.clip(studyArea);
var lc2021 = worldcover2021.clip(studyArea);

// 5. Create urban masks
// Built-up = class 50
var urban2020 = lc2020.eq(50);
var urban2021 = lc2021.eq(50);

// 6. Detect newly urbanized areas
var newUrban = urban2021.and(urban2020.not());

// 7. Display 2020 urban areas
Map.addLayer(
  urban2020.selfMask(),
  {},
  "Urban 2020"
);

// 8. Display 2021 urban areas
Map.addLayer(
  urban2021.selfMask(),
  {},
  "Urban 2021"
);

// 9. Display new urban areas
Map.addLayer(
  newUrban.selfMask(),
  {},
  "Newly Urbanized Areas"
);

// 10. Center map
Map.centerObject(studyArea, 11);
```

---

# 15. Connect to Previous Lessons

This project combines several concepts you've already learned.

### Project 12–13

We learned:

```text
Land cover
     ↓
Built-up class
     ↓
Urban mask
```

### Project 28

We learned:

```text
Earlier image
      ↓
Later image
      ↓
Change detection
```

### Project 38

Now we combine them:

```text
Land cover
    ↓
Built-up extraction
    ↓
Two years
    ↓
Compare
    ↓
Non-urban → Urban
    ↓
Urban expansion
```

So this is really a **land-cover change detection problem**.

---

# 16. Understanding the Four Possible Changes

This table is worth remembering:

| 2020      | 2021      | Meaning                        |
| --------- | --------- | ------------------------------ |
| Non-urban | Non-urban | No urbanization                |
| Non-urban | Urban     | **New urbanization**           |
| Urban     | Non-urban | Possible urban → another class |
| Urban     | Urban     | Existing urban area            |

Our target is only:

```text
Non-urban → Urban
```

---

# 17. Things to Remember

### 1. Built-up = class 50

For ESA WorldCover:

```javascript
lc.eq(50)
```

creates an urban/built-up mask.

### 2. Urban expansion is a change

We're not simply asking:

> Where are cities?

We're asking:

> Where did non-urban land become urban?

### 3. `and()` combines conditions

```javascript
urban2021.and(urban2020.not())
```

means:

> Urban in 2021 **AND** not urban in 2020.

### 4. `not()` reverses a binary mask

```text
1 → 0
0 → 1
```

### 5. `selfMask()` helps visualize the result

It hides the zero values and shows only the pixels satisfying the condition.

---

# 18. Common Beginner Mistakes

### ❌ Mistake 1 — Comparing raw land-cover values

Don't do something like:

```javascript
lc2021.subtract(lc2020)
```

and assume the result directly represents urban growth.

Land-cover classes are **categorical**, not continuous numerical measurements.

For example:

```text
Tree = 10
Cropland = 40
Built-up = 50
```

The difference between 40 and 50 doesn't have a meaningful physical interpretation.

Instead, first convert the categorical data into a meaningful condition:

```javascript
lc.eq(50)
```

Then compare the masks.

---

### ❌ Mistake 2 — Calling all 2021 urban areas "new"

This is wrong:

```javascript
var newUrban = urban2021;
```

That includes cities that already existed in 2020.

We need:

```javascript
var newUrban = urban2021.and(urban2020.not());
```

---

### ❌ Mistake 3 — Assuming every detected change is real construction

Land-cover classification can contain errors.

A pixel classified as built-up in 2021 may not necessarily represent actual new construction.

For research, you'd want to validate the detected changes using higher-resolution imagery or other data.

---

# 19. Practice Project — Urban Expansion

Use the Frankfurt study area.

### Task 1

Display:

* Urban 2020
* Urban 2021
* Newly urbanized areas

Visually compare them.

### Task 2

Find several areas that appear to have changed from:

```text
Non-urban → Urban
```

Then zoom in and inspect the surrounding landscape.

### Task 3 — Challenge

Calculate the **area of newly urbanized land**.

You've already learned how to calculate area in Project 11 using:

```javascript
ee.Image.pixelArea()
```

Your workflow should be:

```text
New urban mask
      ↓
pixelArea()
      ↓
mask
      ↓
sum
      ↓
m²
      ↓
km²
```

Try to write this part yourself.

---

# 20. Conceptual Questions

1. What does **urban expansion** mean in this project?
2. What ESA WorldCover class represents built-up land?
3. Why do we convert land cover into a binary urban mask before comparing years?
4. What does this mean?

```javascript
urban2021.and(urban2020.not())
```

5. Why isn't `urban2021` alone enough to identify new urbanization?
6. Why is subtracting raw land-cover class values not appropriate?
7. What is the difference between:

   * **existing urban area**
   * **newly urbanized area**
8. Why should detected urban expansion ideally be validated with higher-resolution imagery?

### Key idea

> **Urban expansion analysis is a land-cover change problem: identify built-up land in two years, then isolate pixels that changed from non-urban to built-up.**
